# DBSCAN Clustering for Grafana Logs

This notebook performs DBSCAN (Density-Based Spatial Clustering of Applications with Noise) on the extracted features from Grafana logs.

## DBSCAN Algorithm:
- **Type**: Density-based clustering
- **Advantages**: 
  - No need to predefine number of clusters
  - Can find arbitrarily shaped clusters
  - Robust to outliers (identifies noise points)
  - Good for datasets with non-uniform cluster densities
- **Disadvantages**: 
  - Sensitive to parameters (eps and min_samples)
  - Struggles with varying densities
  - Not suitable for high-dimensional data without preprocessing

## Key Parameters:
- **eps**: Maximum distance between two samples to be considered neighbors
- **min_samples**: Minimum number of samples in a neighborhood for a point to be considered a core point

## Steps:
1. Load feature matrices
2. Determine optimal eps using k-distance graph
3. Tune min_samples parameter
4. Perform DBSCAN clustering
5. Evaluate clustering quality
6. Analyze noise points
7. Visualize clusters
8. Analyze cluster characteristics
9. Benchmark performance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import time
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')

print("Libraries imported successfully!")

## 1. Load Feature Matrices

In [ ]:
# Load scaled features
X_scaled = pd.read_csv('features_scaled.csv')
print(f"Scaled features shape: {X_scaled.shape}")

# Load PCA features (DBSCAN works better on reduced dimensions)
X_pca = pd.read_csv('features_pca.csv')
print(f"PCA features shape: {X_pca.shape}")

# Load metadata
metadata = pd.read_csv('metadata.csv')
print(f"Metadata shape: {metadata.shape}")

# For DBSCAN, we'll use PCA features (reduced dimensionality works better)
# Use first N components that explain 95% variance
n_components = 30  # Adjust based on your PCA analysis
X_array = X_pca.iloc[:, :n_components].values

print(f"\nUsing {n_components} PCA components for DBSCAN")
print(f"Data shape: {X_array.shape}")
print("\nData loaded successfully!")

## 2. Determine Optimal eps using k-Distance Graph

In [ ]:
# Calculate k-nearest neighbors distances
# Rule of thumb: k = 2 * dimensions - 1, or use min_samples
k = 4  # Common starting point (2*D for 2D data)

print(f"Computing {k}-nearest neighbors...")
start_time = time.time()
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(X_array)
distances, indices = neighbors_fit.kneighbors(X_array)
computation_time = time.time() - start_time
print(f"Computation completed in {computation_time:.2f} seconds")

# Sort distances to k-th nearest neighbor
distances = np.sort(distances[:, k-1], axis=0)

# Plot k-distance graph
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full k-distance graph
axes[0].plot(distances, linewidth=1)
axes[0].set_xlabel('Data Points (sorted by distance)', fontsize=12)
axes[0].set_ylabel(f'{k}-th Nearest Neighbor Distance', fontsize=12)
axes[0].set_title('k-Distance Graph (Full)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Zoomed k-distance graph (showing elbow more clearly)
percentile_95 = int(len(distances) * 0.95)
axes[1].plot(distances[:percentile_95], linewidth=1)
axes[1].set_xlabel('Data Points (sorted by distance)', fontsize=12)
axes[1].set_ylabel(f'{k}-th Nearest Neighbor Distance', fontsize=12)
axes[1].set_title('k-Distance Graph (Bottom 95%)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dbscan_k_distance_graph.png', dpi=300, bbox_inches='tight')
plt.show()

# Suggest eps values based on k-distance graph
suggested_eps = [
    np.percentile(distances, 90),
    np.percentile(distances, 95),
    np.percentile(distances, 97),
    np.percentile(distances, 99)
]

print("\nSuggested eps values (based on percentiles):")
print(f"  90th percentile: {suggested_eps[0]:.4f}")
print(f"  95th percentile: {suggested_eps[1]:.4f}")
print(f"  97th percentile: {suggested_eps[2]:.4f}")
print(f"  99th percentile: {suggested_eps[3]:.4f}")
print("\nLook for the 'elbow' in the k-distance graph to choose eps.")

## 3. Parameter Tuning: Test Different eps and min_samples

In [ ]:
# Test different parameter combinations
eps_values = [
    suggested_eps[0],
    suggested_eps[1],
    suggested_eps[2],
    suggested_eps[3],
    np.mean(suggested_eps[1:3])  # Average of middle values
]

min_samples_values = [3, 5, 10, 15, 20]

results = []

print("Testing different parameter combinations...")
print("This may take several minutes...\n")

for eps in eps_values:
    for min_samples in min_samples_values:
        print(f"Testing eps={eps:.4f}, min_samples={min_samples}...", end=' ')
        
        try:
            # Perform DBSCAN
            dbscan = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
            labels = dbscan.fit_predict(X_array)
            
            # Count clusters and noise points
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            n_noise = list(labels).count(-1)
            noise_ratio = n_noise / len(labels)
            
            # Calculate metrics only if we have at least 2 clusters and not all noise
            if n_clusters >= 2 and n_noise < len(labels) * 0.9:
                # Filter out noise points for metric calculation
                mask = labels != -1
                if np.sum(mask) > 0:
                    silhouette = silhouette_score(X_array[mask], labels[mask])
                    davies_bouldin = davies_bouldin_score(X_array[mask], labels[mask])
                    calinski_harabasz = calinski_harabasz_score(X_array[mask], labels[mask])
                else:
                    silhouette = davies_bouldin = calinski_harabasz = np.nan
            else:
                silhouette = davies_bouldin = calinski_harabasz = np.nan
            
            results.append({
                'eps': eps,
                'min_samples': min_samples,
                'n_clusters': n_clusters,
                'n_noise': n_noise,
                'noise_ratio': noise_ratio,
                'silhouette': silhouette,
                'davies_bouldin': davies_bouldin,
                'calinski_harabasz': calinski_harabasz
            })
            
            print(f"Clusters: {n_clusters}, Noise: {n_noise} ({noise_ratio*100:.1f}%), Silhouette: {silhouette:.4f}" 
                  if not np.isnan(silhouette) else f"Clusters: {n_clusters}, Noise: {n_noise} ({noise_ratio*100:.1f}%)")
        
        except Exception as e:
            print(f"Failed: {str(e)}")
            continue

results_df = pd.DataFrame(results)
print("\nParameter tuning complete!")

In [ ]:
# Display results sorted by silhouette score
print("\nTop 10 parameter combinations by Silhouette Score:")
print("=" * 100)
top_results = results_df.dropna(subset=['silhouette']).sort_values('silhouette', ascending=False).head(10)
print(top_results.to_string(index=False))

# Visualize parameter tuning results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Number of clusters
pivot_clusters = results_df.pivot_table(values='n_clusters', index='eps', columns='min_samples')
sns.heatmap(pivot_clusters, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0, 0])
axes[0, 0].set_title('Number of Clusters', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('min_samples', fontsize=12)
axes[0, 0].set_ylabel('eps', fontsize=12)

# Noise ratio
pivot_noise = results_df.pivot_table(values='noise_ratio', index='eps', columns='min_samples')
sns.heatmap(pivot_noise, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[0, 1])
axes[0, 1].set_title('Noise Ratio', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('min_samples', fontsize=12)
axes[0, 1].set_ylabel('eps', fontsize=12)

# Silhouette score
pivot_silhouette = results_df.pivot_table(values='silhouette', index='eps', columns='min_samples')
sns.heatmap(pivot_silhouette, annot=True, fmt='.3f', cmap='RdYlGn', ax=axes[1, 0])
axes[1, 0].set_title('Silhouette Score (Higher is Better)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('min_samples', fontsize=12)
axes[1, 0].set_ylabel('eps', fontsize=12)

# Davies-Bouldin Index
pivot_db = results_df.pivot_table(values='davies_bouldin', index='eps', columns='min_samples')
sns.heatmap(pivot_db, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=axes[1, 1])
axes[1, 1].set_title('Davies-Bouldin Index (Lower is Better)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('min_samples', fontsize=12)
axes[1, 1].set_ylabel('eps', fontsize=12)

plt.tight_layout()
plt.savefig('dbscan_parameter_tuning.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Select Optimal Parameters

In [ ]:
# Select best parameters based on silhouette score and reasonable noise ratio
# Filter: noise ratio < 0.3, at least 2 clusters, valid silhouette score
valid_results = results_df[
    (results_df['noise_ratio'] < 0.3) & 
    (results_df['n_clusters'] >= 2) & 
    (~results_df['silhouette'].isna())
]

if len(valid_results) > 0:
    best_params = valid_results.sort_values('silhouette', ascending=False).iloc[0]
else:
    # Fallback: just use best silhouette score
    best_params = results_df.dropna(subset=['silhouette']).sort_values('silhouette', ascending=False).iloc[0]

optimal_eps = best_params['eps']
optimal_min_samples = int(best_params['min_samples'])

print("Selected optimal parameters:")
print(f"  eps: {optimal_eps:.4f}")
print(f"  min_samples: {optimal_min_samples}")
print(f"\nExpected results:")
print(f"  Number of clusters: {int(best_params['n_clusters'])}")
print(f"  Noise points: {int(best_params['n_noise'])} ({best_params['noise_ratio']*100:.1f}%)")
print(f"  Silhouette score: {best_params['silhouette']:.4f}")

## 5. Perform DBSCAN Clustering with Optimal Parameters

In [ ]:
# Perform DBSCAN with optimal parameters
print(f"Performing DBSCAN clustering with eps={optimal_eps:.4f}, min_samples={optimal_min_samples}...")

start_time = time.time()
dbscan_final = DBSCAN(eps=optimal_eps, min_samples=optimal_min_samples, n_jobs=-1)
cluster_labels = dbscan_final.fit_predict(X_array)
clustering_time = time.time() - start_time

print(f"Clustering completed in {clustering_time:.2f} seconds")

# Count clusters and noise
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)

print(f"\nCluster distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    if cluster_id == -1:
        print(f"  Noise: {count} samples ({count/len(cluster_labels)*100:.2f}%)")
    else:
        print(f"  Cluster {cluster_id}: {count} samples ({count/len(cluster_labels)*100:.2f}%)")

print(f"\nTotal clusters found: {n_clusters}")
print(f"Total noise points: {n_noise} ({n_noise/len(cluster_labels)*100:.2f}%)")

## 6. Evaluate Clustering Quality

In [ ]:
# Calculate evaluation metrics (excluding noise points)
mask = cluster_labels != -1

if np.sum(mask) > 0 and len(set(cluster_labels[mask])) >= 2:
    silhouette = silhouette_score(X_array[mask], cluster_labels[mask])
    davies_bouldin = davies_bouldin_score(X_array[mask], cluster_labels[mask])
    calinski_harabasz = calinski_harabasz_score(X_array[mask], cluster_labels[mask])
else:
    silhouette = davies_bouldin = calinski_harabasz = np.nan

print("=" * 80)
print("DBSCAN CLUSTERING EVALUATION METRICS")
print("=" * 80)
print(f"\nParameters:")
print(f"  - eps: {optimal_eps:.4f}")
print(f"  - min_samples: {optimal_min_samples}")
print(f"\nClustering Results:")
print(f"  - Number of clusters: {n_clusters}")
print(f"  - Number of samples: {len(cluster_labels):,}")
print(f"  - Number of noise points: {n_noise:,} ({n_noise/len(cluster_labels)*100:.2f}%)")
print(f"  - Clustering time: {clustering_time:.2f} seconds")
print(f"\nQuality Metrics (excluding noise):")
if not np.isnan(silhouette):
    print(f"  - Silhouette Score: {silhouette:.4f} (range: [-1, 1], higher is better)")
    print(f"  - Davies-Bouldin Index: {davies_bouldin:.4f} (lower is better)")
    print(f"  - Calinski-Harabasz Score: {calinski_harabasz:.2f} (higher is better)")
else:
    print(f"  - Metrics not available (insufficient clusters or too much noise)")
print("\n" + "=" * 80)

# Store metrics for comparison
dbscan_metrics = {
    'algorithm': 'DBSCAN',
    'eps': optimal_eps,
    'min_samples': optimal_min_samples,
    'n_clusters': n_clusters,
    'n_noise': n_noise,
    'noise_ratio': n_noise / len(cluster_labels),
    'silhouette_score': silhouette if not np.isnan(silhouette) else None,
    'davies_bouldin_index': davies_bouldin if not np.isnan(davies_bouldin) else None,
    'calinski_harabasz_score': calinski_harabasz if not np.isnan(calinski_harabasz) else None,
    'clustering_time': clustering_time,
    'n_samples': len(cluster_labels)
}

## 7. Analyze Noise Points

In [ ]:
# Analyze characteristics of noise points
if n_noise > 0:
    noise_indices = np.where(cluster_labels == -1)[0]
    cluster_indices = np.where(cluster_labels != -1)[0]
    
    print("Noise Point Analysis:")
    print("=" * 80)
    
    # Compare noise vs cluster points on metadata
    noise_metadata = metadata.iloc[noise_indices]
    cluster_metadata = metadata.iloc[cluster_indices]
    
    print("\nValue statistics comparison:")
    print("Noise points:")
    print(f"  Mean: {noise_metadata['value'].mean():.2f}")
    print(f"  Median: {noise_metadata['value'].median():.2f}")
    print(f"  Std: {noise_metadata['value'].std():.2f}")
    
    print("\nCluster points:")
    print(f"  Mean: {cluster_metadata['value'].mean():.2f}")
    print(f"  Median: {cluster_metadata['value'].median():.2f}")
    print(f"  Std: {cluster_metadata['value'].std():.2f}")
    
    print("\nTop panels in noise:")
    print(noise_metadata['panel_title'].value_counts().head(5))
    
    print("\nTop services in noise:")
    print(noise_metadata['service'].value_counts().head(5))
    
    print("\n" + "=" * 80)
else:
    print("No noise points detected.")

## 8. Visualize Clusters

In [ ]:
# Visualize clusters using first 3 PCA components
X_pca_full = X_pca.values

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: PC1 vs PC2
# Separate noise points
mask_noise = cluster_labels == -1
mask_cluster = cluster_labels != -1

if np.sum(mask_cluster) > 0:
    scatter1 = axes[0].scatter(X_pca_full[mask_cluster, 0], X_pca_full[mask_cluster, 1], 
                               c=cluster_labels[mask_cluster], cmap='viridis', alpha=0.6, s=30, label='Clusters')
if np.sum(mask_noise) > 0:
    axes[0].scatter(X_pca_full[mask_noise, 0], X_pca_full[mask_noise, 1], 
                    c='red', alpha=0.3, s=10, marker='x', label='Noise')

axes[0].set_xlabel('First Principal Component', fontsize=12)
axes[0].set_ylabel('Second Principal Component', fontsize=12)
axes[0].set_title('DBSCAN Clusters (PC1 vs PC2)', fontsize=14, fontweight='bold')
axes[0].legend()
if np.sum(mask_cluster) > 0:
    plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Plot 2: PC2 vs PC3
if np.sum(mask_cluster) > 0:
    scatter2 = axes[1].scatter(X_pca_full[mask_cluster, 1], X_pca_full[mask_cluster, 2], 
                               c=cluster_labels[mask_cluster], cmap='viridis', alpha=0.6, s=30, label='Clusters')
if np.sum(mask_noise) > 0:
    axes[1].scatter(X_pca_full[mask_noise, 1], X_pca_full[mask_noise, 2], 
                    c='red', alpha=0.3, s=10, marker='x', label='Noise')

axes[1].set_xlabel('Second Principal Component', fontsize=12)
axes[1].set_ylabel('Third Principal Component', fontsize=12)
axes[1].set_title('DBSCAN Clusters (PC2 vs PC3)', fontsize=14, fontweight='bold')
axes[1].legend()
if np.sum(mask_cluster) > 0:
    plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.savefig('dbscan_clusters_pca.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3D visualization
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

if np.sum(mask_cluster) > 0:
    scatter = ax.scatter(X_pca_full[mask_cluster, 0], X_pca_full[mask_cluster, 1], X_pca_full[mask_cluster, 2],
                         c=cluster_labels[mask_cluster], cmap='viridis', alpha=0.6, s=20, label='Clusters')
if np.sum(mask_noise) > 0:
    ax.scatter(X_pca_full[mask_noise, 0], X_pca_full[mask_noise, 1], X_pca_full[mask_noise, 2],
               c='red', alpha=0.3, s=10, marker='x', label='Noise')

ax.set_xlabel('PC1', fontsize=12)
ax.set_ylabel('PC2', fontsize=12)
ax.set_zlabel('PC3', fontsize=12)
ax.set_title('DBSCAN Clusters in 3D PCA Space', fontsize=14, fontweight='bold')
ax.legend()
if np.sum(mask_cluster) > 0:
    plt.colorbar(scatter, ax=ax, label='Cluster', shrink=0.6)

plt.tight_layout()
plt.savefig('dbscan_clusters_3d.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Cluster Characteristics Analysis

In [ ]:
# Add cluster labels to metadata
metadata['cluster'] = cluster_labels

# Analyze cluster characteristics (excluding noise)
if n_clusters > 0:
    print("Cluster Characteristics:\n")
    print("=" * 80)
    
    for cluster_id in range(n_clusters):
        cluster_data = metadata[metadata['cluster'] == cluster_id]
        
        print(f"\nCluster {cluster_id} (n={len(cluster_data)}):")
        print("-" * 80)
        
        # Top panels
        print("  Top 5 Panels:")
        top_panels = cluster_data['panel_title'].value_counts().head(5)
        for panel, count in top_panels.items():
            print(f"    - {panel}: {count} ({count/len(cluster_data)*100:.1f}%)")
        
        # Top services
        print("\n  Top 5 Services:")
        top_services = cluster_data['service'].value_counts().head(5)
        for service, count in top_services.items():
            print(f"    - {service}: {count} ({count/len(cluster_data)*100:.1f}%)")
        
        # Value statistics
        print("\n  Value Statistics:")
        print(f"    - Mean: {cluster_data['value'].mean():.2f}")
        print(f"    - Median: {cluster_data['value'].median():.2f}")
        print(f"    - Std: {cluster_data['value'].std():.2f}")
        print(f"    - Min: {cluster_data['value'].min():.2f}")
        print(f"    - Max: {cluster_data['value'].max():.2f}")
    
    print("\n" + "=" * 80)
else:
    print("No clusters found (all points classified as noise).")

In [ ]:
# Visualize cluster characteristics
if n_clusters > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Cluster size distribution (including noise)
    cluster_sizes = metadata['cluster'].value_counts().sort_index()
    colors = ['red' if idx == -1 else 'steelblue' for idx in cluster_sizes.index]
    axes[0, 0].bar(range(len(cluster_sizes)), cluster_sizes.values, color=colors)
    axes[0, 0].set_xlabel('Cluster ID (-1 = Noise)', fontsize=12)
    axes[0, 0].set_ylabel('Number of Samples', fontsize=12)
    axes[0, 0].set_title('Cluster Size Distribution', fontsize=14, fontweight='bold')
    axes[0, 0].set_xticks(range(len(cluster_sizes)))
    axes[0, 0].set_xticklabels(cluster_sizes.index)
    axes[0, 0].grid(True, alpha=0.3)
    
    # Value distribution by cluster (excluding noise)
    metadata[metadata['cluster'] != -1].boxplot(column='value', by='cluster', ax=axes[0, 1])
    axes[0, 1].set_xlabel('Cluster ID', fontsize=12)
    axes[0, 1].set_ylabel('Value', fontsize=12)
    axes[0, 1].set_title('Value Distribution by Cluster', fontsize=14, fontweight='bold')
    plt.sca(axes[0, 1])
    plt.xticks(rotation=0)
    
    # Panel distribution across clusters (excluding noise)
    cluster_data = metadata[metadata['cluster'] != -1]
    if len(cluster_data) > 0:
        top_panels = cluster_data['panel_title'].value_counts().head(8).index
        panel_cluster_counts = pd.crosstab(cluster_data['panel_title'], cluster_data['cluster'])
        if len(top_panels) > 0:
            panel_cluster_counts.loc[top_panels].plot(kind='bar', stacked=True, ax=axes[1, 0], colormap='viridis')
            axes[1, 0].set_xlabel('Panel Title', fontsize=12)
            axes[1, 0].set_ylabel('Count', fontsize=12)
            axes[1, 0].set_title('Top Panels Distribution Across Clusters', fontsize=14, fontweight='bold')
            axes[1, 0].legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.sca(axes[1, 0])
            plt.xticks(rotation=45, ha='right')
        
        # Service distribution across clusters (excluding noise)
        top_services = cluster_data['service'].value_counts().head(8).index
        service_cluster_counts = pd.crosstab(cluster_data['service'], cluster_data['cluster'])
        if len(top_services) > 0:
            service_cluster_counts.loc[top_services].plot(kind='bar', stacked=True, ax=axes[1, 1], colormap='viridis')
            axes[1, 1].set_xlabel('Service', fontsize=12)
            axes[1, 1].set_ylabel('Count', fontsize=12)
            axes[1, 1].set_title('Top Services Distribution Across Clusters', fontsize=14, fontweight='bold')
            axes[1, 1].legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.sca(axes[1, 1])
            plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig('dbscan_cluster_characteristics.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Cannot visualize cluster characteristics (no clusters found).")

## 10. Save Results

In [ ]:
# Save cluster assignments
results_df = metadata.copy()
results_df.to_csv('dbscan_cluster_assignments.csv', index=False)
print("Cluster assignments saved to: dbscan_cluster_assignments.csv")

# Save metrics
import json
with open('dbscan_metrics.json', 'w') as f:
    json.dump(dbscan_metrics, f, indent=2)
print("Metrics saved to: dbscan_metrics.json")

# Save model
import pickle
with open('dbscan_model.pkl', 'wb') as f:
    pickle.dump(dbscan_final, f)
print("Model saved to: dbscan_model.pkl")

# Save parameter tuning results
results_df_params = pd.DataFrame(results)
results_df_params.to_csv('dbscan_parameter_tuning_results.csv', index=False)
print("Parameter tuning results saved to: dbscan_parameter_tuning_results.csv")

print("\nDBSCAN clustering complete!")